In [ ]:
import pandas as pd
import numpy as np

: 

In [ ]:
df_Master = pd.read_parquet('df_Master_Final.parquet')

In [ ]:
df_Master['hour'] = df_Master['created_at'].dt.hour
df_Master['month'] = df_Master['created_at'].dt.month
df_Master['day_name'] = df_Master['created_at'].dt.day_name()
df_Master['is_weekend'] = np.where(
    df_Master['day_name'].isin(['Saturday', 'Sunday']),
    'Weekend',
    'Weekday'
)
df_Master['member_status'] = np.where(df_Master['user_id'].notna(), 'Member', 'Guest')
df_Master['is_voucher_used'] = np.where(
    df_Master['voucher_id'].notna(),
    'Voucher',
    'No Voucher'
)
df_Master['month_name'] = (
    df_Master['created_at']
    .dt.month_name()
)
df_Master['transaction_period'] = np.select(
    [
        df_Master['hour'].between(5, 10),
        df_Master['hour'].between(11, 15),
        df_Master['hour'].between(16, 19),
        df_Master['hour'].between(20, 23)
    ],
    [
        'Morning',
        'Afternoon',
        'Evening',
        'Night'
    ],
    default='Late Night'
)
df_Master['discount_ratio'] = (
    df_Master['discount_applied'] / (df_Master['original_amount'] + 1e-6)
)

In [ ]:
print(df_Master.columns)

In [ ]:
df_Master = df_Master.rename(columns={
    'category_x': 'menu_category',
    'category_y': 'payment_category'
})
df_Master = df_Master.drop(
    columns=['method_id', 'Non-Member', 'Member'],
    errors='ignore'
)

In [ ]:
df_transaction_features = (
    df_Master
    .groupby('transaction_id')
    .agg({
        'quantity': 'sum',
        'final_amount': 'nunique',
        'discount_applied': 'first',
        'is_weekend': 'first',
        'is_voucher_used': 'first',
        'hour': 'first',
        'month_name': 'first',
        'day_name': 'first',
        'city': 'first',
        'method_name': 'first',
        'payment_category': 'first',
        'member_status': 'first',
        'created_at': 'first',
        'user_id': 'first',
        'transaction_period': 'first'
    })
    .reset_index()
    .rename(columns={
        'quantity': 'basket_size'
    })
)

In [ ]:
snapshot_date = (
    df_transaction_features['created_at'].max()
    + pd.Timedelta(days=1)
)

rfm_table = (
    df_transaction_features[
        df_transaction_features['user_id'].notna()
    ]
    .groupby('user_id')
    .agg(
        Recency=('created_at', lambda x: (snapshot_date - x.max()).days),
        Frequency=('transaction_id', 'count'),
        Monetary=('final_amount', 'sum')
    )
    .reset_index()
)
rfm_table['is_repeat_customer'] = np.where(
    rfm_table['Frequency'] > 1,
    'Repeat Customer',
    'One-Time Customer'
)

In [ ]:
df_Master.to_parquet(
    'df_Master_FE.parquet',
    index=False
)

df_transaction_features.to_parquet(
    'df_transaction_features.parquet',
    index=False
)

rfm_table.to_parquet(
    'df_rfm.parquet',
    index=False
)